# Stress test — ML pipeline (branching, sweeps, dedup, chunking)

A realistic ML workflow: ingest → two cleaning branches → features → a hyperparameter
sweep of models → evaluation. Exercises dedup (re-running identical steps), chunking
(near-identical feature matrices), and rich metadata. **Watch for the cells flagged
✅ — they demonstrate findings from the original 0.1.x stress report, re-verified on
the SQLite backend.**

In [1]:
import shutil
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import ancestree
from ancestree import LineageStore

SCRATCH = Path(tempfile.mkdtemp(prefix="ancestree-stress-"))  # safe to delete


def store(name, **kw):
    return LineageStore(SCRATCH / name, **kw)


def node_count(s):
    # Nodes are rows now, not folders: count them with a query.
    return len(s.find())


print("ancestree", getattr(ancestree, "__version__", "?"))

ancestree 0.1.0


## Build the pipeline

In [2]:
RULES = {
    "ingest": [None],
    "clean": ["ingest"],
    "features": ["clean"],
    "model": ["features"],
    "eval": ["model"],
}
s = store("ml", rules=RULES, gen_triggers=["ingest"], dedup=True, chunk=True)

rng = np.random.default_rng(0)
N = 60_000
raw = pd.DataFrame(
    {
        "f1": rng.normal(size=N),
        "f2": rng.normal(size=N),
        "f3": rng.normal(size=N),
        "label": rng.integers(0, 2, N),
    }
)
with s.create_node(step_type="ingest") as ingest:
    raw.to_csv(ingest / "raw.csv", index=False)
    ingest.add_meta("rows", int(len(raw)), group="Stats")
print("ingest:", ingest.node_id)

ingest: fc033079


In [3]:
# Two near-identical cleaning branches (great for sub-file chunk sharing)
clean_a_df = raw.dropna().reset_index(drop=True)
with s.create_node(step_type="clean", parent=ingest) as clean_a:
    clean_a_df.to_csv(clean_a / "clean.csv", index=False)
    clean_a.add_meta("strategy", "dropna")

clean_b_df = raw.copy()
clean_b_df["f1"] = clean_b_df["f1"].clip(-2, 2)
with s.create_node(step_type="clean", parent=ingest) as clean_b:
    clean_b_df.to_csv(clean_b / "clean.csv", index=False)
    clean_b.add_meta("strategy", "clip")

with s.create_node(step_type="features", parent=clean_a) as feats:
    X = np.c_[clean_a_df["f1"], clean_a_df["f2"], clean_a_df["f1"] * clean_a_df["f2"]]
    np.save(feats / "X.npy", X)
    feats.add_meta("n_features", int(X.shape[1]), group="Stats")
print("built:", clean_a.node_id, clean_b.node_id, feats.node_id)

built: 27884581 df11ae89 3a3f11ba


## Hyperparameter sweep — each config is a distinct model node

In [4]:
configs = [{"lr": lr, "depth": d} for lr in (0.01, 0.1) for d in (3, 5, 7)]
for cfg in configs:
    with s.create_node(step_type="model", parent=feats) as m:
        w = rng.normal(size=3)
        (m / "model.npy").write_bytes(w.tobytes())
        acc = float(rng.uniform(0.8, 0.95))
        m.add_meta("config", cfg, group="Config")
        m.add_meta("accuracy", round(acc, 4), group="Metrics")
models = s.find(step_type="model")
print(f"{len(models)} model nodes from the sweep")

6 model nodes from the sweep


In [5]:
s.generate_web_graph()

PosixPath('/var/folders/xf/n_m7ztrx4x577r3935n_1m1w0000gn/T/ancestree-stress-om98zjq7/ml/interactive_pipeline.html')

## ✅ Finding #1 — numpy / pandas scalars in `add_meta`

In early 0.1.x, `np.int64`, `np.bool_`, `pd.Timestamp`, sets and arrays raised
`TypeError` at the **end of the `with` block**. The fix carried straight over to the
rebuild: `add_meta` **coerces** them to native Python types and **warns** you it did so;
anything still not serialisable is rejected **at the `add_meta` call** (not at block
exit). The cell below shows both.

In [6]:
import warnings


def try_meta(label, value):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        try:
            with s.create_node(step_type="eval", parent=models[0]) as ev:
                ev.add_meta("metric", value)
            warned = any("coerced" in str(x.message) for x in w)
            got = s.get(ev.node_id).metadata["metric"]["value"]
            print(f"  OK    {label:16s} -> {got!r:16} (coerced/warned={warned})")
        except Exception as e:
            print(f"  REJECTED {label:13s} -> {type(e).__name__} at call time")


try_meta("python int", 5)  # native, no warning
try_meta("np.float64", raw["f1"].mean())  # native (float subclass)
try_meta("np.int64 sum", raw["label"].sum())  # coerced -> int, warns
try_meta("np.bool_", np.bool_(True))  # coerced -> bool, warns
try_meta("pd.Timestamp", pd.Timestamp.now())  # coerced -> isoformat str
try_meta("np.array", np.arange(3))  # coerced -> list
try_meta("bytes (uncoercible)", b"\x00\x01")  # rejected at the add_meta call

  OK    python int       -> 5                (coerced/warned=False)
  OK    np.float64       -> 0.00033895957312896123 (coerced/warned=False)
  OK    np.int64 sum     -> 30216            (coerced/warned=True)
  OK    np.bool_         -> True             (coerced/warned=True)


  OK    pd.Timestamp     -> '2026-07-09T07:55:47.662547' (coerced/warned=True)
  OK    np.array         -> [0, 1, 2]        (coerced/warned=True)
  REJECTED bytes (uncoercible) -> TypeError at call time


In [7]:
# A rejected value raises at the add_meta call (before anything persists), so
# nothing is left behind: the node count is unchanged and the scratch area is
# empty (in 0.1.x this check looked for orphan node DIRECTORIES; nodes are
# rows now, and the transient scratch is the only filesystem footprint).
print("nodes after the rejection:", node_count(s))
leftovers = list((s.root / ".scratch").iterdir()) if (s.root / ".scratch").exists() else []
print("scratch leftovers:", len(leftovers))

nodes after the rejection: 16
scratch leftovers: 0


## Re-run the whole pipeline — dedup reuses identical nodes

In [8]:
before = node_count(s)
# Re-run ingest (deterministic content) — should dedup, not duplicate
rng2 = np.random.default_rng(0)
raw2 = pd.DataFrame(
    {
        "f1": rng2.normal(size=N),
        "f2": rng2.normal(size=N),
        "f3": rng2.normal(size=N),
        "label": rng2.integers(0, 2, N),
    }
)
with s.create_node(step_type="ingest") as ingest2:
    raw2.to_csv(ingest2 / "raw.csv", index=False)
    ingest2.add_meta("rows", int(len(raw2)), group="Stats")
print("re-run ingest id == original?", ingest2.node_id == ingest.node_id)
print(
    "node count before:", before, "after:", node_count(s), "(no new node = deduped)"
)

re-run ingest id == original? True
node count before: 16 after: 16 (no new node = deduped)


## Chunk efficiency — sub-file sharing across the two clean branches

0.1.x walked the on-disk chunk pool and the per-node `.artifacts.json` manifests to
compute this. Both live in the database now, so `store.stats()` answers directly.

In [9]:
stats = s.stats()
pool = stats["chunk_stored_bytes"]
logical = stats["logical_bytes"]
print(f"pool in the store: {pool / 1e6:.2f} MB   logical artifact bytes: {logical / 1e6:.2f} MB")
print(f"compression + dedup factor: {logical / max(pool, 1):.1f}x")
print(f"(store.stats() reports the same as dedup_ratio: {stats['dedup_ratio']})")

pool in the store: 4.79 MB   logical artifact bytes: 12.36 MB
compression + dedup factor: 2.6x
(store.stats() reports the same as dedup_ratio: 2.58)


In [10]:
s.generate_web_graph()  # works with packed artifacts (uses logical names)
print("web graph written to", s.root / "interactive_pipeline.html")

s.close()
shutil.rmtree(SCRATCH)

web graph written to /var/folders/xf/n_m7ztrx4x577r3935n_1m1w0000gn/T/ancestree-stress-om98zjq7/ml/interactive_pipeline.html
